In [1]:
import os
import sys
import zipfile
import requests
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split



In [2]:
# ====================== DOWNLOAD MOVIELENS 1M ======================
def download_movielens():
    url = "https://files.grouplens.org/datasets/movielens/ml-1m.zip"
    zip_path = "ml-1m.zip"
    extract_path = "ml-1m"
    
    if not os.path.exists(extract_path):
        print("Downloading MovieLens 1M dataset...")
        r = requests.get(url, stream=True)
        with open(zip_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=8192):
                f.write(chunk)
        
        print("Extracting...")
        with zipfile.ZipFile(zip_path) as z:
            z.extractall(".")
        os.remove(zip_path)
        print("Download complete!")
    else:
        print("MovieLens 1M already downloaded.")

download_movielens()



MovieLens 1M already downloaded.


In [8]:
# ====================== LOAD & PREPROCESS ======================
ratings = pd.read_csv(
    "ml-1m/ratings.dat",
    sep="::",
    engine="python",
    names=["userId", "movieId", "rating", "timestamp"]
)

# ====================== Custom Dataset ======================

TTE_ids = pd.read_json(f"TTE_user_split_uf90.json")

def no_diplicate_films(tp, columns):
  film_list = []
  for c in columns:
    for m_list in tp[c]:
      for m in m_list:
        film_list.append(m)
  return list(dict.fromkeys(film_list))

def filter_triplets(tp):
    usercount = len(tp)
    film_list = []
    itemcount = len(no_diplicate_films(tp, ['train_h', 'train_c', 'test_h', 'test_c', 'eval_h', 'eval_c']))
    return tp, usercount, itemcount

TTE_ids , user_activity, item_popularity = filter_triplets(TTE_ids)
print(user_activity, item_popularity)

tr_users = TTE_ids['user_id']
#te_users = TTE_ids['user_id']

#tr_films = no_diplicate_films(TTE_ids, ['train_h', 'train_c', 'test_h', 'test_c'])
#te_films = no_diplicate_films(TTE_ids, ['eval_h', 'eval_c'])
all_films = no_diplicate_films(TTE_ids, ['train_h', 'train_c', 'test_h', 'test_c', 'eval_h', 'eval_c'])

"""
train_plays = ratings.loc[ratings['userId'].isin(tr_users)]
train_plays = train_plays.loc[train_plays['movieId'].isin(tr_films)]

test_plays = ratings.loc[ratings['userId'].isin(te_users)]
test_plays = test_plays.loc[test_plays['movieId'].isin(te_films)]
"""

train_input_plays = pd.DataFrame()
valid_input_plays = pd.DataFrame()
test_input_plays = pd.DataFrame()

train_output_plays = pd.DataFrame()
valid_output_plays = pd.DataFrame()
test_output_plays = pd.DataFrame()

for idx, row in TTE_ids.iterrows():
    user_films = ratings.loc[ratings['userId'].isin([row['user_id']])]
    for c in ['train_h', 'train_c', 'test_h', 'test_c', 'eval_h', 'eval_c']:
        user_tr = user_films.loc[user_films['movieId'].isin(row[c])]
        match c:
            case 'train_h':
                train_input_plays = pd.concat([train_input_plays, user_tr], ignore_index=True)
            case 'train_c':
                train_output_plays = pd.concat([train_output_plays, user_tr], ignore_index=True)
            case 'test_h':
                valid_input_plays = pd.concat([valid_input_plays, user_tr], ignore_index=True)
            case 'test_c':
                valid_output_plays = pd.concat([valid_output_plays, user_tr], ignore_index=True)
            case 'eval_h':
                test_input_plays = pd.concat([test_input_plays, user_tr], ignore_index=True)
            case 'eval_c':
                test_output_plays = pd.concat([test_output_plays, user_tr], ignore_index=True)

print(len(train_input_plays))
print(len(test_output_plays))

user_ids = tr_users
movie_ids = all_films
user_map = {uid: idx for idx, uid in enumerate(user_ids)}
movie_map = {mid: idx for idx, mid in enumerate(movie_ids)}

n_users = len(user_ids)
n_items = len(movie_ids)
print(f"Users: {n_users}, Items: {n_items}")

# ====================== Matrix Dataset ======================

# Build interaction matrix (dense is fine for ML-1M: ~6k x 3.7k)
def build_matrix(plays):
    interactions = np.zeros((n_users, n_items), dtype=np.float32)
    print()
    for n_iter, row in plays.iterrows():
        if row["userId"] in user_map.keys() and row["movieId"] in movie_map.keys():
            u = user_map[row["userId"]]
            i = movie_map[row["movieId"]]
            interactions[u, i] = 1.0
    return interactions

"""
interactions_te = np.zeros((n_users, n_items), dtype=np.float32)
print()
for n_iter, row in test_plays.iterrows():
    if n_iter%100 == 0:
        sys.stdout.write(f"\rTest Iter: {n_iter}/{len(test_plays)}")
        sys.stdout.flush()
    if row["userId"] in user_map.keys() and row["movieId"] in te_films:
        u = user_map[row["userId"]]
        i = movie_map[row["movieId"]]
        interactions_te[u, i] = 1.0
"""

train_matrix_i = build_matrix(train_input_plays)
train_matrix_o = build_matrix(train_output_plays)
valid_matrix_i = build_matrix(valid_input_plays)
valid_matrix_o = build_matrix(valid_output_plays)
test_matrix_i = build_matrix(test_input_plays)
test_matrix_o = build_matrix(test_output_plays)

print(len(train_matrix_i), len(train_matrix_i[0]))


4430 3387
44300
44300
Users: 4430, Items: 3387






4430 3387


In [13]:
# ====================== DATASET ======================
class UserDataset(Dataset):
    def __init__(self, input_matrix, output_matrix):
        self.input_data  = torch.from_numpy(input_matrix).float()
        self.output_data = torch.from_numpy(output_matrix).float()
   
    def __len__(self):
        return len(self.input_data)
   
    def __getitem__(self, idx):
        return self.input_data[idx], self.output_data[idx]   # returns (input, target)

#train_dataset = UserDataset(train_matrix)
#train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True, num_workers=2)

train_dataset = UserDataset(train_matrix_i, train_matrix_o)
valid_dataset   = UserDataset(valid_matrix_i,   valid_matrix_o)
test_dataset  = UserDataset(test_matrix_i,  test_matrix_o)

train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True,  num_workers=2)
valid_loader = DataLoader(valid_dataset,   batch_size=256, shuffle=False, num_workers=2)

In [10]:
# ====================== MULT-VAE MODEL ======================
class MultVAE(nn.Module):
    def __init__(self, n_items, hidden_dim=600, latent_dim=200, dropout=0.5):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(n_items, hidden_dim),
            nn.Tanh(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Dropout(dropout)
        )
        self.fc_mu = nn.Linear(hidden_dim, latent_dim)
        self.fc_logvar = nn.Linear(hidden_dim, latent_dim)
        
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.Tanh(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, n_items)
        )
    
    def encode(self, x):
        h = self.encoder(x)
        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        return mu, logvar
    
    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std
    
    def decode(self, z):
        return self.decoder(z)
    
    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        logits = self.decode(z)
        return logits, mu, logvar

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = MultVAE(n_items).to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)



In [11]:
# ====================== LOSS (Multinomial + KL) ======================
def mult_vae_loss(logits, target, mu, logvar, anneal=1.0):
    # Reconstruction: negative multinomial log-likelihood
    log_softmax = torch.log_softmax(logits, dim=1)
    recon_loss = -torch.sum(target * log_softmax, dim=1)
    
    # KL divergence (standard Gaussian prior)
    kl_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp(), dim=1)
    
    return torch.mean(recon_loss + anneal * kl_loss)



In [15]:
# ====================== TRAINING ======================
epochs = 50
anneal_steps = 10  # gradually increase KL weight
best_loss = float("inf")
best_val_loss = float("inf")
patience = 1000          # early stopping patience
counter = 0

loss_history = []

for epoch in range(epochs):
    model.train()
    total_loss = 0.0
    anneal = min(1.0, (epoch + 1) / anneal_steps)  # simple linear annealing
    
    for input_batch, output_batch in train_loader:
        input_batch  = input_batch.to(device)
        output_batch = output_batch.to(device)
        
        optimizer.zero_grad()
        logits, mu, logvar = model(input_batch)
        loss = mult_vae_loss(logits, output_batch, mu, logvar, anneal)
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)  # stability
        optimizer.step()
        
        total_loss += loss.item()
        
    avg_train_loss = total_loss / len(train_loader)
    loss_history.append(avg_train_loss)
    
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for input_batch, output_batch in valid_loader:
            input_batch  = input_batch.to(device)
            output_batch = output_batch.to(device)
            logits, mu, logvar = model(input_batch)
            loss = mult_vae_loss(logits, output_batch, mu, logvar, anneal)
            val_loss += loss.item()
   
    avg_val_loss = val_loss / len(valid_loader)
    
    print(f"Epoch {epoch+1:2d}/{epochs} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Anneal: {anneal:.2f}")
    
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        counter = 0
        torch.save(model.state_dict(), "multvae_movielens.pth")
    else:
        counter += 1
        if counter >= patience:
            print("Early stopping triggered!")
            break

print("Training finished! Model saved as multvae_movielens.pth")



Epoch  1/50 | Train Loss: 74.8198 | Val Loss: 75.5918 | Anneal: 0.10
Epoch  2/50 | Train Loss: 74.9871 | Val Loss: 75.7770 | Anneal: 0.20
Epoch  3/50 | Train Loss: 75.1790 | Val Loss: 75.9937 | Anneal: 0.30
Epoch  4/50 | Train Loss: 75.2515 | Val Loss: 76.1066 | Anneal: 0.40
Epoch  5/50 | Train Loss: 75.2343 | Val Loss: 76.0512 | Anneal: 0.50
Epoch  6/50 | Train Loss: 75.2013 | Val Loss: 76.0026 | Anneal: 0.60
Epoch  7/50 | Train Loss: 74.8735 | Val Loss: 75.8717 | Anneal: 0.70
Epoch  8/50 | Train Loss: 74.6650 | Val Loss: 75.6027 | Anneal: 0.80
Epoch  9/50 | Train Loss: 74.3506 | Val Loss: 75.4356 | Anneal: 0.90
Epoch 10/50 | Train Loss: 74.0770 | Val Loss: 75.3224 | Anneal: 1.00
Epoch 11/50 | Train Loss: 73.7230 | Val Loss: 75.3105 | Anneal: 1.00
Epoch 12/50 | Train Loss: 73.5010 | Val Loss: 75.3312 | Anneal: 1.00
Epoch 13/50 | Train Loss: 73.1956 | Val Loss: 75.3541 | Anneal: 1.00
Epoch 14/50 | Train Loss: 73.1829 | Val Loss: 75.5697 | Anneal: 1.00
Epoch 15/50 | Train Loss: 72.9713 

In [31]:
# ====================== RECOMMENDATION FUNCTION ======================
def recommend_for_user(model, user_idx, train_interactions, k=10):
    model.eval()
    with torch.no_grad():
        user_vec = torch.tensor(train_matrix[user_idx]).unsqueeze(0).float().to(device)
        logits, _, _ = model(user_vec)
        scores = logits.squeeze().cpu().numpy()
        
        # Mask already seen items
        seen = train_interactions[user_idx] > 0
        scores[seen] = -np.inf
        
        top_k_idx = np.argsort(scores)[-k:][::-1]
        return top_k_idx

# Load movie titles for nice output
movies = pd.read_csv(
    "ml-1m/movies.dat",
    sep="::",
    engine="python",
    names=["movieId", "title", "genres"],
    encoding="ISO-8859-1"
)
movie_title_map = {}
for _, row in movies.iterrows():
    if row["movieId"] in movie_map:
        movie_title_map[movie_map[row["movieId"]]] = row["title"]

# Example recommendation for user 0 (change user_idx as you like)
user_idx = 0
top_movies_idx = recommend_for_user(model, user_idx, train_matrix, k=10)
print("\n=== Recommended movies for user", user_idx, "===")
for rank, item_idx in enumerate(top_movies_idx, 1):
    title = movie_title_map.get(item_idx, f"Unknown (ID {item_idx})")
    print(f"{rank}. {title}")




=== Recommended movies for user 0 ===
1. Field of Dreams (1989)
2. Little Mermaid, The (1989)
3. Natural, The (1984)
4. Beauty and the Beast (1991)
5. Lion King, The (1994)
6. Quiz Show (1994)
7. Edward Scissorhands (1990)
8. Chariots of Fire (1981)
9. Boogie Nights (1997)
10. Gandhi (1982)


In [33]:
# ====================== SIMPLE EVALUATION (Recall@10 & NDCG@10) ======================
@torch.no_grad()
def evaluate(model, train_mat, test_mat, k=10):
    model.eval()
    recalls = []
    ndcgs = []
    
    for u in range(n_users):
        if np.sum(test_mat[u]) == 0:
            continue
            
        user_vec = torch.tensor(train_mat[u]).unsqueeze(0).float().to(device)
        logits = model(user_vec)[0].squeeze().cpu().numpy()
        
        # Mask train items
        scores = logits.copy()
        scores[train_mat[u] > 0] = -np.inf
        
        ranked = np.argsort(scores)[::-1]
        true_items = set(np.where(test_mat[u] > 0)[0])
        
        # Recall@K
        hits = len(set(ranked[:k]) & true_items)
        recall = hits / len(true_items)
        recalls.append(recall)
        
        # NDCG@K
        dcg = sum(1 / np.log2(i + 2) for i, item in enumerate(ranked[:k]) if item in true_items)
        idcg = sum(1 / np.log2(i + 2) for i in range(min(k, len(true_items))))
        ndcg = dcg / idcg if idcg > 0 else 0
        ndcgs.append(ndcg)
    
    return np.mean(recalls), np.mean(ndcgs)

rec, ndcg = evaluate(model, train_matrix, test_matrix, k=100)
print(f"\nEvaluation on held-out test set:")
print(f"Recall@10: {rec:.4f}")
print(f"NDCG@10 : {ndcg:.4f}")


Evaluation on held-out test set:
Recall@10: 0.1463
NDCG@10 : 0.0956
